# EDA — Bank Credit Card Transactions (`creditcard.csv`)

**Objective:** Explore the anonymized PCA-transformed bank transaction dataset. This notebook covers:
1. Data loading and profiling
2. Class imbalance quantification
3. Time and Amount distributions
4. PCA feature exploration (V1–V28)
5. Correlations with fraud target

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_creditcard, basic_info
from src.preprocessing import drop_duplicates, handle_missing

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete')

## 1. Load and Profile

In [ ]:
cc_df = load_creditcard()
basic_info(cc_df, 'creditcard')
cc_df.describe().T.style.background_gradient(cmap='Blues')

## 2. Class Imbalance

In [ ]:
class_counts = cc_df['Class'].value_counts()
class_pct = cc_df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(pd.DataFrame({'count': class_counts, 'pct': class_pct.round(4)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
class_counts.plot.bar(ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Transaction Count by Class')
axes[0].set_xticklabels(['Legit (0)', 'Fraud (1)'], rotation=0)
axes[0].set_ylabel('Count')

axes[1].pie(class_counts, labels=['Legit', 'Fraud'], autopct='%1.3f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

## 3. Time and Amount Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time distribution
sns.histplot(cc_df['Time'], bins=100, ax=axes[0][0], color='steelblue')
axes[0][0].set_title('Transaction Time Distribution (seconds)')

# Amount distribution (log-scale)
cc_df['Amount'].apply(lambda x: np.log1p(x)).pipe(
    lambda s: sns.histplot(s, bins=80, ax=axes[0][1], color='slateblue'))
axes[0][1].set_title('log1p(Amount) Distribution')

# Amount by class
sns.boxplot(x='Class', y='Amount', data=cc_df, ax=axes[1][0],
            palette={0: 'steelblue', 1: 'tomato'})
axes[1][0].set_xticklabels(['Legit', 'Fraud'])
axes[1][0].set_title('Amount vs Class')

# Time by class
sns.boxplot(x='Class', y='Time', data=cc_df, ax=axes[1][1],
            palette={0: 'steelblue', 1: 'tomato'})
axes[1][1].set_xticklabels(['Legit', 'Fraud'])
axes[1][1].set_title('Time vs Class')

plt.tight_layout()
plt.show()

## 4. PCA Feature Distributions by Class

In [ ]:
v_cols = [c for c in cc_df.columns if c.startswith('V')]
fraud = cc_df[cc_df['Class'] == 1]
legit = cc_df[cc_df['Class'] == 0]

# Plot distributions for top 12 V features
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()

for i, col in enumerate(v_cols[:12]):
    axes[i].hist(legit[col], bins=60, alpha=0.5, color='steelblue', label='Legit', density=True)
    axes[i].hist(fraud[col], bins=60, alpha=0.7, color='tomato', label='Fraud', density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle('PCA Feature Distributions by Class (Legit vs Fraud)', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Correlation of all features with Class
corr_with_class = cc_df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(6, 10))
corr_with_class.plot.barh(color=corr_with_class.map(lambda x: 'tomato' if x > 0 else 'steelblue'))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Fraud Class')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

print('Top positive correlations with fraud:')
print(corr_with_class.tail(10))
print('\nTop negative correlations with fraud:')
print(corr_with_class.head(10))

## 6. Cleaning and Save

In [ ]:
print('Before cleaning:', cc_df.shape)
cc_df = drop_duplicates(cc_df)
cc_df = handle_missing(cc_df, strategy='drop')
print('After cleaning:', cc_df.shape)

cc_df.to_csv('../data/processed/creditcard_cleaned.csv', index=False)
print('Saved → data/processed/creditcard_cleaned.csv')

## Summary

**Key Findings:**
- Credit card data is *extremely* imbalanced — fraud < 0.2% of transactions.
- Several V features (V14, V4, V11, V12) show very distinct distributions between legit and fraud.
- Fraud transactions tend to have lower `Amount` values.
- No missing values. One set of exact duplicates removed.

**Next step:** Modeling notebook → `modeling.ipynb`